In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import re

In [2]:
data = pd.read_csv("synthetic_medical_triage.csv")
print("Dataset Info:")
print(f"Shape: {data.shape}")
print(f"\nColumns: {data.columns.tolist()}")
print(f"\nMissing values:\n{data.isnull().sum()}")
print(f"\nTriage level distribution:\n{data['triage_level'].value_counts()}")


Dataset Info:
Shape: (18000, 10)

Columns: ['age', 'heart_rate', 'systolic_blood_pressure', 'oxygen_saturation', 'body_temperature', 'pain_level', 'chronic_disease_count', 'previous_er_visits', 'arrival_mode', 'triage_level']

Missing values:
age                        0
heart_rate                 0
systolic_blood_pressure    0
oxygen_saturation          0
body_temperature           0
pain_level                 0
chronic_disease_count      0
previous_er_visits         0
arrival_mode               0
triage_level               0
dtype: int64

Triage level distribution:
triage_level
0    9924
1    4484
2    2701
3     891
Name: count, dtype: int64


In [3]:
reader = PdfReader("NDMA.pdf")
text = ""
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print(f"Extracted {len(text)} characters from PDF")

Extracted 22528 characters from PDF


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,  # Smaller chunks for more precise retrieval
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

chunks = splitter.split_text(text)
print(f"Created {len(chunks)} chunks")

# Cell 5: Create Embeddings and Vector Store
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks, show_progress_bar=True)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print(f"Created FAISS index with {index.ntotal} vectors")


Created 93 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Created FAISS index with 93 vectors


In [5]:
def retrieve_relevant_chunks(query, chunks, index, model, k=7):
    """Retrieve relevant chunks with better scoring"""
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, k)
    
    # Get chunks and their similarity scores
    retrieved = []
    for i, idx in enumerate(indices[0]):
        # Convert L2 distance to similarity score (0-1)
        similarity = 1 / (1 + distances[0][i])
        retrieved.append({
            'text': chunks[idx],
            'score': similarity,
            'index': idx
        })

    retrieved.sort(key=lambda x: x['score'], reverse=True)
    return retrieved


In [6]:
print("Loading better model...")
model_name = "microsoft/DialoGPT-medium"  # Better than distilgpt2
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
llm_model = AutoModelForCausalLM.from_pretrained(model_name)

# Create generator with better parameters
generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=250,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,
    do_sample=True
)

Loading better model...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

c:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--microsoft--DialoGPT-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'repetition_penalty', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [7]:
def create_triage_prompt(query, retrieved_chunks):
    """Create a structured prompt for triage assessment"""
    
    # Format retrieved context
    context = "\n\n".join([f"[Document {i+1}]: {chunk['text'][:300]}" 
                          for i, chunk in enumerate(retrieved_chunks[:3])])
    
    prompt = f"""You are an expert emergency medical triage assistant. Based on the provided guidelines and medical knowledge, analyze the query and provide a structured triage assessment.
    MEDICAL GUIDELINES:
{context}

PATIENT QUERY: {query}

Based on the symptoms described, provide a complete triage assessment in the following format:

TRIAGE LEVEL: [0 (Non-urgent)/1 (Less urgent)/2 (Urgent)/3 (Emergency)]
PRIORITY: [High/Medium/Low]

IMMEDIATE ACTIONS:
• [First action]
• [Second action]
• [Third action]

RESPONSIBLE DEPARTMENT: [Emergency/ICU/General Ward/OPD]

URGENCY: [Immediate (within minutes)/Very Urgent (within 1 hour)/Urgent (within 4 hours)/Routine]

REASONING: [Brief explanation of why this triage level was assigned]

RISK FACTORS: [List key risk factors based on vital signs or symptoms]

FOLLOW-UP: [Recommended follow-up timeline]

ASSESSMENT:
"""
    return prompt



In [8]:
def parse_triage_response(response_text, query):
    """Extract structured information from LLM response"""
    
    # Default values
    triage_assessment = {
        'query': query,
        'triage_level': 'Unknown',
        'priority': 'Unknown',
        'immediate_actions': [],
        'department': 'Unknown',
        'urgency': 'Unknown',
        'reasoning': '',
        'risk_factors': '',
        'follow_up': ''
    }
    
     # Extract sections using regex
    triage_match = re.search(r'TRIAGE LEVEL:\s*(\d+)', response_text)
    if triage_match:
        triage_assessment['triage_level'] = triage_match.group(1)
    
    priority_match = re.search(r'PRIORITY:\s*(\w+)', response_text)
    if priority_match:
        triage_assessment['priority'] = priority_match.group(1)

    # Extract actions (lines starting with •)
    actions = re.findall(r'•\s*(.*?)(?=\n•|\n\n|\Z)', response_text, re.DOTALL)
    triage_assessment['immediate_actions'] = [a.strip() for a in actions[:3]]
    
    dept_match = re.search(r'RESPONSIBLE DEPARTMENT:\s*(.*?)(?=\n)', response_text)
    if dept_match:
        triage_assessment['department'] = dept_match.group(1).strip()
    
    urgency_match = re.search(r'URGENCY:\s*(.*?)(?=\n)', response_text)
    if urgency_match:
        triage_assessment['urgency'] = urgency_match.group(1).strip()
    
    reasoning_match = re.search(r'REASONING:\s*(.*?)(?=\n\n|\Z)', response_text, re.DOTALL)
    if reasoning_match:
        triage_assessment['reasoning'] = reasoning_match.group(1).strip()
    
    return triage_assessment

# Cell 10: Main Triage Function
def triage_assessment(query, chunks, index, model, generator, k=7):
    """Complete triage assessment pipeline"""
    
    # 1. Retrieve relevant chunks
    retrieved = retrieve_relevant_chunks(query, chunks, index, model, k)
    
    print(f"\n📋 Retrieved {len(retrieved)} relevant sections")
    print(f"Top match score: {retrieved[0]['score']:.3f}")
    
    # 2. Create prompt
    prompt = create_triage_prompt(query, retrieved)
    
    # 3. Generate response
    try:
        response = generator(prompt, max_new_tokens=300, pad_token_id=tokenizer.eos_token_id)
        full_response = response[0]['generated_text']
        
        # Extract only the new part (after the prompt)
        new_text = full_response[len(prompt):].strip()
        
        # 4. Parse response
        assessment = parse_triage_response(new_text, query)
        
        return assessment, retrieved, new_text
        
    except Exception as e:
        print(f"Error generating response: {e}")
        return None, retrieved, None

In [ ]:
test_queries = [
    "earthquake victim severe injury bleeding from head, unconscious",
    "elderly patient with chest pain and difficulty breathing",
    "child with fever 102°F and rash",
    "car accident victim with broken leg, conscious and stable"
]

for query in test_queries:
    print("\n" + "="*60)
    print(f"QUERY: {query}")
    print("="*60)

    assessment, retrieved, raw_response = triage_assessment(
        query, chunks, index, model, generator
    )
    
    if assessment:
        print("\n📊 TRIAGE ASSESSMENT:")
        print(f"Triage Level: {assessment['triage_level']}")
        print(f"Priority: {assessment['priority']}")
        print(f"Department: {assessment['department']}")
        print(f"Urgency: {assessment['urgency']}")
        print(f"\nImmediate Actions:")
        for action in assessment['immediate_actions']:
            print(f"  • {action}")
        print(f"\nReasoning: {assessment['reasoning'][:200]}...")

    print("\n" + "-"*40)
    

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUERY: earthquake victim severe injury bleeding from head, unconscious

📋 Retrieved 7 relevant sections
Top match score: 0.396


In [ ]:
def interactive_triage():
    """Run interactive triage assessment"""
    print("\n" + "="*50)
    print("MEDICAL TRIAGE ASSISTANT")
    print("="*50)
    print("Enter patient symptoms (or 'quit' to exit)")
    
    while True:
        print("\n" + "-"*30)
        query = input("Symptoms: ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            break

        if not query:
            continue
        
        assessment, _, _ = triage_assessment(query, chunks, index, model, generator)
        
        if assessment:
            print("\n" + "="*40)
            print("TRIAGE RESULTS")
            print("="*40)
            print(f"Level {assessment['triage_level']} - {assessment['priority']} Priority")
            print(f"Department: {assessment['department']}")
            print(f"See within: {assessment['urgency']}")
            print("\nActions:")
            for action in assessment['immediate_actions']:
                print(f"  • {action}")


In [ ]:
interactive_triage()

In [ ]:
print("\n📊 TESTING WITH DATASET SAMPLES")
print("="*50)

# Get some sample patients from dataset
sample_patients = data.sample(3)
for idx, patient in sample_patients.iterrows():
    # Create query from patient data
    query = f"Patient age {patient['age']:.0f}, heart rate {patient['heart_rate']:.0f}, "
    query += f"BP {patient['systolic_blood_pressure']:.0f}, oxygen {patient['oxygen_saturation']:.0f}%, "
    query += f"temp {patient['body_temperature']:.1f}°C, pain level {patient['pain_level']}, "
    query += f"arrived by {patient['arrival_mode']}"

    print(f"\nPatient {idx}:")
    print(f"Actual triage level: {patient['triage_level']}")
    print(f"Query: {query}")
    
    assessment, _, _ = triage_assessment(query, chunks, index, model, generator)
    
    if assessment:
        print(f"Predicted triage: {assessment['triage_level']}")
        print(f"Match: {'✓' if str(patient['triage_level']) == assessment['triage_level'] else '✗'}")


In [1]:
import secrets
print(secrets.token_hex(32))

d7df7177bfa6e6a38ec03f63fe64c2f4eefba58c6d6fd19343a1c94b1a0e25cd
